In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

# Le préfixe r permet de conserver les antislashs du chemin Windows.
DOSSIER_DONNEES = Path(
    r"C:\Users\David KUNYIMA\OneDrive\Desktop\Memoire_CNN_Attention\archive\chest_xray"
)

if not DOSSIER_DONNEES.is_dir():
    raise FileNotFoundError(
        f"Le dossier est introuvable : {DOSSIER_DONNEES}"
    )

extensions_images = {".jpeg", ".jpg", ".png", ".bmp", ".tif", ".tiff"}
resultats = []

for ensemble in ["train", "val", "test"]:
    for classe in ["NORMAL", "PNEUMONIA"]:
        dossier = DOSSIER_DONNEES / ensemble / classe
        present = dossier.is_dir()

        nombre = (
            sum(
                1 for fichier in dossier.iterdir()
                if fichier.is_file()
                and fichier.suffix.lower() in extensions_images
            )
            if present else 0
        )

        resultats.append({
            "Ensemble": ensemble,
            "Classe": classe,
            "Dossier présent": present,
            "Nombre d'images": nombre
        })

tableau = pd.DataFrame(resultats)

display(tableau)
print("Total des fichiers images :", tableau["Nombre d'images"].sum())

if not tableau["Dossier présent"].all():
    print("Attention : certains sous-dossiers attendus sont absents.")
elif (tableau["Nombre d'images"] == 0).any():
    print("Attention : au moins un sous-dossier ne contient aucune image.")
else:
    print("Les six sous-dossiers sont présents et contiennent des images.")

,Ensemble,Classe,Dossier présent,Nombre d'images
0,train,NORMAL,True,1341
1,train,PNEUMONIA,True,3875
2,val,NORMAL,True,8
3,val,PNEUMONIA,True,8
4,test,NORMAL,True,234
5,test,PNEUMONIA,True,390


Total des fichiers images : 5856
Les six sous-dossiers sont présents et contiennent des images.


In [2]:
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
import pandas as pd
from IPython.display import display

DOSSIER_DONNEES = Path(
    r"C:\Users\David KUNYIMA\OneDrive\Desktop\Memoire_CNN_Attention\archive\chest_xray"
)

extensions_images = {".jpeg", ".jpg", ".png", ".bmp", ".tif", ".tiff"}

images_valides = []
erreurs = []

for ensemble in ["train", "val", "test"]:
    for classe in ["NORMAL", "PNEUMONIA"]:
        dossier = DOSSIER_DONNEES / ensemble / classe

        if not dossier.is_dir():
            raise FileNotFoundError(f"Dossier absent : {dossier}")

        fichiers = sorted(
            fichier for fichier in dossier.iterdir()
            if fichier.is_file()
            and fichier.suffix.lower() in extensions_images
        )

        for fichier in tqdm(fichiers, desc=f"{ensemble} / {classe}"):
            try:
                # Vérification de la structure du fichier.
                with Image.open(fichier) as image:
                    image.verify()

                # Réouverture et lecture complète des pixels.
                with Image.open(fichier) as image:
                    image.load()
                    largeur, hauteur = image.size

                    images_valides.append({
                        "ensemble": ensemble,
                        "classe": classe,
                        "nom": fichier.name,
                        "chemin": str(fichier),
                        "largeur": largeur,
                        "hauteur": hauteur,
                        "mode": image.mode
                    })

            except Exception as erreur:
                erreurs.append({
                    "chemin": str(fichier),
                    "erreur": str(erreur)
                })

inventaire = pd.DataFrame(images_valides)
images_en_erreur = pd.DataFrame(erreurs)

print("\nImages lisibles :", len(inventaire))
print("Images avec une erreur :", len(images_en_erreur))

if not inventaire.empty:
    print("\nDimensions des images, en pixels :")
    display(
        inventaire[["largeur", "hauteur"]]
        .agg(["min", "max"])
        .rename(index={"min": "Minimum", "max": "Maximum"})
    )

    print("\nModes des images :")
    display(inventaire["mode"].value_counts().rename("Nombre d'images"))

if not images_en_erreur.empty:
    print("\nPremières erreurs à examiner :")
    display(images_en_erreur.head(10))

c:\Users\David KUNYIMA\anaconda3\envs\memoire_cnn\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
test / PNEUMONIA: 100%|██████████| 390/390 [00:02<00:00, 132.28it/s]


Images lisibles : 5856
Images avec une erreur : 0

Dimensions des images, en pixels :


,largeur,hauteur
Minimum,384,127
Maximum,2916,2713



Modes des images :


mode
L      5573
RGB     283
Name: Nombre d'images, dtype: int64

In [3]:
import hashlib
from PIL import Image
from tqdm import tqdm
from IPython.display import display

if "inventaire" not in globals():
    raise RuntimeError(
        "Exécute d'abord la cellule de vérification des images."
    )

def empreinte_image(chemin):
    """Calcule une empreinte à partir des dimensions et des pixels."""
    with Image.open(chemin) as image:
        image_rgb = image.convert("RGB")

        empreinte = hashlib.sha256()
        empreinte.update(
            f"{image_rgb.width},{image_rgb.height}|RGB|".encode("utf-8")
        )
        empreinte.update(image_rgb.tobytes())

        return empreinte.hexdigest()

# Une empreinte permet de comparer les contenus des images.
empreintes = [
    empreinte_image(chemin)
    for chemin in tqdm(
        inventaire["chemin"],
        desc="Recherche des doublons"
    )
]

inventaire = inventaire.copy()
inventaire["empreinte"] = empreintes

# On conserve ici toutes les occurrences des images répétées.
doublons = inventaire[
    inventaire.duplicated("empreinte", keep=False)
].copy()

print("\nImages analysées :", len(inventaire))
print("Contenus uniques :", inventaire["empreinte"].nunique())

if doublons.empty:
    print("Aucun doublon exact détecté.")
else:
    groupes_doublons = (
        doublons.groupby("empreinte")
        .agg(
            nombre_images=("chemin", "size"),
            nombre_ensembles=("ensemble", "nunique"),
            ensembles=(
                "ensemble",
                lambda valeurs: ", ".join(sorted(set(valeurs)))
            ),
            nombre_classes=("classe", "nunique")
        )
        .reset_index()
    )

    print("Groupes de doublons :", len(groupes_doublons))
    print("Fichiers appartenant à ces groupes :", len(doublons))
    print(
        "Copies supplémentaires :",
        len(doublons) - len(groupes_doublons)
    )
    print(
        "Groupes présents dans plusieurs ensembles :",
        (groupes_doublons["nombre_ensembles"] > 1).sum()
    )
    print(
        "Groupes avec des classes contradictoires :",
        (groupes_doublons["nombre_classes"] > 1).sum()
    )

    print("\nRépartition des groupes de doublons :")
    display(
        groupes_doublons.groupby("ensembles")
        .size()
        .rename("Nombre de groupes")
        .to_frame()
    )

    print("\nExemples de fichiers concernés :")
    display(
        doublons.sort_values(["empreinte", "chemin"])[
            ["ensemble", "classe", "nom", "empreinte"]
        ].head(12)
    )

Recherche des doublons: 100%|██████████| 5856/5856 [03:25<00:00, 28.46it/s]



Images analysées : 5856
Contenus uniques : 5824
Groupes de doublons : 30
Fichiers appartenant à ces groupes : 62
Copies supplémentaires : 32
Groupes présents dans plusieurs ensembles : 0
Groupes avec des classes contradictoires : 0

Répartition des groupes de doublons :


,Nombre de groupes
ensembles,
test,6
train,24



Exemples de fichiers concernés :


,ensemble,classe,nom,empreinte
5326,test,NORMAL,NORMAL2-IM-0095-0001.jpeg,046f6a5715c8679899412144f954cc6f4cb98f2dfde503...
5327,test,NORMAL,NORMAL2-IM-0096-0001.jpeg,046f6a5715c8679899412144f954cc6f4cb98f2dfde503...
2686,train,PNEUMONIA,person1628_bacteria_4297.jpeg,12a3735f35ccd4bf19662fd0ae2f967cecf781f23a8b5b...
2687,train,PNEUMONIA,person1628_bacteria_4298.jpeg,12a3735f35ccd4bf19662fd0ae2f967cecf781f23a8b5b...
2673,train,PNEUMONIA,person1619_bacteria_4267.jpeg,12defef979f2884b28ead126cdb763fe64ca223cf90746...
2674,train,PNEUMONIA,person1619_bacteria_4268.jpeg,12defef979f2884b28ead126cdb763fe64ca223cf90746...
5683,test,PNEUMONIA,person1661_virus_2872.jpeg,148679f85cc778c1b15874cce39da4a0cf90d103c05d83...
5684,test,PNEUMONIA,person1661_virus_2873.jpeg,148679f85cc778c1b15874cce39da4a0cf90d103c05d83...
5349,test,NORMAL,NORMAL2-IM-0173-0001-0001.jpeg,15b831f2925a12dd605de9edd7a66e847b6becfa8d8329...
5350,test,NORMAL,NORMAL2-IM-0173-0001-0002.jpeg,15b831f2925a12dd605de9edd7a66e847b6becfa8d8329...


In [4]:
import pandas as pd
from IPython.display import display

if "inventaire" not in globals() or "empreinte" not in inventaire.columns:
    raise RuntimeError(
        "Exécute d'abord les cellules de vérification et de recherche des doublons."
    )

# Conserver tous les noms et chemins pour les vérifications suivantes.
inventaire_complet = inventaire.copy()

# Vérifier que le résultat correspond bien au cas que nous traitons.
controle = inventaire_complet.groupby("empreinte").agg(
    nombre_ensembles=("ensemble", "nunique"),
    nombre_classes=("classe", "nunique")
)

if (controle["nombre_ensembles"] > 1).any():
    raise ValueError(
        "Des doublons traversent plusieurs ensembles : examen nécessaire."
    )

if (controle["nombre_classes"] > 1).any():
    raise ValueError(
        "Des images identiques portent des classes différentes."
    )

# Trier pour conserver toujours la même occurrence à chaque exécution.
inventaire_trie = inventaire_complet.sort_values(
    ["ensemble", "classe", "nom", "chemin"]
)

copies_supplementaires = inventaire_trie.duplicated(
    subset="empreinte",
    keep="first"
)

inventaire_sans_doublons = (
    inventaire_trie.loc[~copies_supplementaires]
    .copy()
    .reset_index(drop=True)
)

copies_ecartees = (
    inventaire_trie.loc[copies_supplementaires]
    .copy()
    .reset_index(drop=True)
)

# Comparaison avant et après.
avant = inventaire_complet.groupby(["ensemble", "classe"]).size()
apres = inventaire_sans_doublons.groupby(["ensemble", "classe"]).size()

bilan = pd.concat(
    [avant.rename("Avant"), apres.rename("Après")],
    axis=1
).fillna(0).astype(int)

bilan["Copies écartées"] = bilan["Avant"] - bilan["Après"]
display(bilan)

print("Nombre initial :", len(inventaire_complet))
print("Copies écartées de la liste :", len(copies_ecartees))
print("Images retenues :", len(inventaire_sans_doublons))
print("Aucun fichier image n'a été supprimé.")

# Examiner les conventions de nommage avant d'identifier les patients.
print("\nExemples de noms par ensemble et par classe :")

for (ensemble, classe), groupe in inventaire_complet.groupby(
    ["ensemble", "classe"]
):
    print(f"\n{ensemble} / {classe}")
    for nom in sorted(groupe["nom"])[:5]:
        print("  ", nom)

Avant  Après  Copies écartées
ensemble classe                                  
test     NORMAL       234    231                3
         PNEUMONIA    390    387                3
train    NORMAL      1341   1340                1
         PNEUMONIA   3875   3850               25
val      NORMAL         8      8                0
         PNEUMONIA      8      8                0

Nombre initial : 5856
Copies écartées de la liste : 32
Images retenues : 5824
Aucun fichier image n'a été supprimé.

Exemples de noms par ensemble et par classe :

test / NORMAL
   IM-0001-0001.jpeg
   IM-0003-0001.jpeg
   IM-0005-0001.jpeg
   IM-0006-0001.jpeg
   IM-0007-0001.jpeg

test / PNEUMONIA
   person100_bacteria_475.jpeg
   person100_bacteria_477.jpeg
   person100_bacteria_478.jpeg
   person100_bacteria_479.jpeg
   person100_bacteria_480.jpeg

train / NORMAL
   IM-0115-0001.jpeg
   IM-0117-0001.jpeg
   IM-0119-0001.jpeg
   IM-0122-0001.jpeg
   IM-0125-0001.jpeg

train / PNEUMONIA
   person1000_bacteria_2931.jpeg
   person1000_virus_1681.jpeg
   person1001_bacteria_2932.jpeg
   person1002_bacteria_2933.jpeg
   person1003_bacteria_2934.jpeg

val / NORMAL
   NORMAL2-IM-1427-0001.jpeg
   NORMAL2-IM-1430-0001.jpeg
   NORMAL2-IM-1431-0001.jpeg
   NORMAL2-IM-1436-0001.jpeg
   NORMAL2-IM-1437-0001.jpeg

val / PNEUMONIA
   person1946_bacteria_4874.jpeg
   person1946_bacteria_4875.jpeg


In [5]:
import re
from itertools import combinations
from IPython.display import display

if "inventaire_complet" not in globals():
    raise RuntimeError("Exécute d'abord la cellule précédente.")

def extraire_identifiant(nom):
    # Exemple : person100_bacteria_475.jpeg → person100
    correspondance = re.match(r"^(person\d+)_", nom, flags=re.I)
    if correspondance:
        return correspondance.group(1).lower()

    # Exemples :
    # IM-0115-0001.jpeg → IM-0115
    # NORMAL2-IM-1427-0001.jpeg → NORMAL2-IM-1427
    correspondance = re.match(
        r"^((?:NORMAL2-)?IM-\d+)-",
        nom,
        flags=re.I
    )
    if correspondance:
        return correspondance.group(1).upper()

    # Ne pas inventer d'identifiant pour un nom non reconnu.
    return None

audit_identifiants = inventaire_complet.copy()
audit_identifiants["id_presume"] = (
    audit_identifiants["nom"].apply(extraire_identifiant)
)

non_reconnus = audit_identifiants[
    audit_identifiants["id_presume"].isna()
]

print("Noms non reconnus :", len(non_reconnus))

if not non_reconnus.empty:
    display(non_reconnus[["ensemble", "classe", "nom"]].head(10))

print("\nNombre d'identifiants présumés par ensemble et classe :")
display(
    audit_identifiants.groupby(["ensemble", "classe"])["id_presume"]
    .nunique()
    .rename("Identifiants présumés")
    .to_frame()
)

print("\nIdentifiants communs entre les ensembles :")

for ensemble_a, ensemble_b in combinations(["train", "val", "test"], 2):
    ids_a = set(
        audit_identifiants.loc[
            audit_identifiants["ensemble"] == ensemble_a,
            "id_presume"
        ].dropna()
    )
    ids_b = set(
        audit_identifiants.loc[
            audit_identifiants["ensemble"] == ensemble_b,
            "id_presume"
        ].dropna()
    )

    communs = sorted(ids_a & ids_b)
    print(f"{ensemble_a} / {ensemble_b} : {len(communs)}")

    if communs:
        print("Exemples :", communs[:8])

# Une image identique peut être enregistrée sous plusieurs identifiants.
ids_par_image = (
    audit_identifiants.groupby("empreinte")["id_presume"].nunique()
)
empreintes_ambigues = ids_par_image[ids_par_image > 1].index

print(
    "\nGroupes d'images identiques associés à plusieurs identifiants :",
    len(empreintes_ambigues)
)

if len(empreintes_ambigues) > 0:
    display(
        audit_identifiants[
            audit_identifiants["empreinte"].isin(empreintes_ambigues)
        ].sort_values(["empreinte", "nom"])[
            ["ensemble", "classe", "nom", "id_presume"]
        ].head(12)
    )

Noms non reconnus : 0

Nombre d'identifiants présumés par ensemble et classe :


Identifiants présumés
ensemble classe                          
test     NORMAL                       225
         PNEUMONIA                    202
train    NORMAL                      1211
         PNEUMONIA                   1635
val      NORMAL                         8
         PNEUMONIA                      7


Identifiants communs entre les ensembles :
train / val : 0
train / test : 170
Exemples : ['person1', 'person10', 'person100', 'person101', 'person102', 'person103', 'person104', 'person108']
val / test : 0

Groupes d'images identiques associés à plusieurs identifiants : 1


,ensemble,classe,nom,id_presume
5326,test,NORMAL,NORMAL2-IM-0095-0001.jpeg,NORMAL2-IM-0095
5327,test,NORMAL,NORMAL2-IM-0096-0001.jpeg,NORMAL2-IM-0096


In [6]:
from IPython.display import display

if "audit_identifiants" not in globals():
    raise RuntimeError("Exécute d'abord la cellule d'analyse des identifiants.")

# 1. Vérifier si tous les noms ont été reconnus.
non_reconnus = audit_identifiants[
    audit_identifiants["id_presume"].isna()
]

print("Noms non reconnus :", len(non_reconnus))

if not non_reconnus.empty:
    display(non_reconnus[["ensemble", "classe", "nom"]].head(10))

# 2. Afficher le groupe dont les copies portent des identifiants différents.
ids_par_image = (
    audit_identifiants.groupby("empreinte")["id_presume"].nunique()
)
empreintes_ambigues = ids_par_image[ids_par_image > 1].index

print("\nImages identiques portant plusieurs identifiants :")

cas_ambigus = audit_identifiants[
    audit_identifiants["empreinte"].isin(empreintes_ambigues)
]

print(
    cas_ambigus[
        ["ensemble", "classe", "nom", "id_presume"]
    ].sort_values(["id_presume", "nom"]).to_string(index=False)
)

# 3. Mesurer combien d'images retenues sont concernées par le chevauchement.
donnees_auditees = inventaire_sans_doublons.merge(
    audit_identifiants[["chemin", "id_presume"]],
    on="chemin",
    how="left",
    validate="one_to_one"
)

ids_test = set(
    audit_identifiants.loc[
        audit_identifiants["ensemble"] == "test", "id_presume"
    ].dropna()
)

chevauchement = donnees_auditees[
    donnees_auditees["ensemble"].isin(["train", "val"])
    & donnees_auditees["id_presume"].isin(ids_test)
]

print("\nImages hors test partageant un identifiant avec le test :")
display(
    chevauchement.groupby(["ensemble", "classe"])
    .size()
    .rename("Nombre d'images")
    .to_frame()
)

print("\nExemples pour l'identifiant person100 :")
print(
    donnees_auditees.loc[
        donnees_auditees["id_presume"] == "person100",
        ["ensemble", "classe", "nom"]
    ].head(12).to_string(index=False)
)

Noms non reconnus : 0

Images identiques portant plusieurs identifiants :
ensemble classe                       nom      id_presume
    test NORMAL NORMAL2-IM-0095-0001.jpeg NORMAL2-IM-0095
    test NORMAL NORMAL2-IM-0096-0001.jpeg NORMAL2-IM-0096

Images hors test partageant un identifiant avec le test :


,,Nombre d'images
ensemble,classe,
train,PNEUMONIA,453



Exemples pour l'identifiant person100 :
ensemble    classe                         nom
    test PNEUMONIA person100_bacteria_475.jpeg
    test PNEUMONIA person100_bacteria_477.jpeg
    test PNEUMONIA person100_bacteria_478.jpeg
    test PNEUMONIA person100_bacteria_479.jpeg
    test PNEUMONIA person100_bacteria_480.jpeg
    test PNEUMONIA person100_bacteria_481.jpeg
    test PNEUMONIA person100_bacteria_482.jpeg
   train PNEUMONIA    person100_virus_184.jpeg


In [7]:
import pandas as pd
from itertools import combinations
from sklearn.model_selection import train_test_split
from IPython.display import display

if "audit_identifiants" not in globals():
    raise RuntimeError("Exécute d'abord les cellules précédentes.")

audit = audit_identifiants.copy()

if audit["id_presume"].isna().any():
    raise ValueError("Des identifiants restent à examiner.")

# 1. Relier les identifiants associés à une même image.
# On utilise l'inventaire complet pour inclure les copies écartées.
parents = {identifiant: identifiant
           for identifiant in audit["id_presume"].unique()}

def racine(identifiant):
    while parents[identifiant] != identifiant:
        parents[identifiant] = parents[parents[identifiant]]
        identifiant = parents[identifiant]
    return identifiant

def reunir(a, b):
    racine_a, racine_b = racine(a), racine(b)
    if racine_a != racine_b:
        parents[max(racine_a, racine_b)] = min(racine_a, racine_b)

for _, groupe in audit.groupby("empreinte"):
    identifiants = sorted(groupe["id_presume"].unique())
    for identifiant in identifiants[1:]:
        reunir(identifiants[0], identifiant)

audit["groupe"] = audit["id_presume"].map(racine)

# 2. Ajouter les groupes à la liste sans doublons.
donnees = inventaire_sans_doublons.merge(
    audit[["chemin", "id_presume", "groupe"]],
    on="chemin",
    how="left",
    validate="one_to_one"
)

assert donnees["groupe"].notna().all()

# 3. Conserver le test et exclure ses groupes des autres ensembles.
groupes_test = set(
    audit.loc[audit["ensemble"] == "test", "groupe"]
)

test_df = donnees[donnees["ensemble"] == "test"].copy()

hors_test = donnees[
    donnees["ensemble"].isin(["train", "val"])
].copy()

exclusions_df = hors_test[
    hors_test["groupe"].isin(groupes_test)
].copy()
exclusions_df["motif"] = "Groupe également présent dans le test"

developpement = hors_test[
    ~hors_test["groupe"].isin(groupes_test)
].copy()

# 4. Préparer une ligne par groupe, puis répartir les groupes.
classes_par_groupe = developpement.groupby("groupe")["classe"].nunique()

if (classes_par_groupe > 1).any():
    raise ValueError(
        "Un groupe contient plusieurs classes : examen nécessaire."
    )

table_groupes = (
    developpement[["groupe", "classe"]]
    .drop_duplicates()
    .sort_values("groupe")
    .reset_index(drop=True)
)

groupes_train, groupes_val = train_test_split(
    table_groupes,
    test_size=0.20,
    random_state=42,
    stratify=table_groupes["classe"]
)

train_df = developpement[
    developpement["groupe"].isin(groupes_train["groupe"])
].copy()

val_df = developpement[
    developpement["groupe"].isin(groupes_val["groupe"])
].copy()

# Conserver "ensemble" comme provenance et ajouter l'affectation finale.
train_df["ensemble_final"] = "train"
val_df["ensemble_final"] = "val"
test_df["ensemble_final"] = "test"

repartition = pd.concat(
    [train_df, val_df, test_df], ignore_index=True
)

# 5. Vérifier la séparation des groupes et des contenus.
ensembles = {"train": train_df, "val": val_df, "test": test_df}

for nom_a, nom_b in combinations(ensembles, 2):
    a, b = ensembles[nom_a], ensembles[nom_b]

    assert set(a["groupe"]).isdisjoint(b["groupe"]), (
        f"Groupes communs entre {nom_a} et {nom_b}"
    )
    assert set(a["empreinte"]).isdisjoint(b["empreinte"]), (
        f"Images identiques entre {nom_a} et {nom_b}"
    )

assert len(repartition) + len(exclusions_df) == len(donnees)
assert repartition["chemin"].is_unique

# 6. Afficher le bilan.
bilan = pd.crosstab(
    repartition["ensemble_final"], repartition["classe"]
).reindex(["train", "val", "test"])

bilan["Total images"] = bilan.sum(axis=1)
bilan["Nombre de groupes"] = (
    repartition.groupby("ensemble_final")["groupe"].nunique()
)

display(bilan)

print("Images écartées par précaution :", len(exclusions_df))
print("Images d'apprentissage + validation :", len(developpement))
print("Aucun groupe commun entre les trois ensembles.")
print("Aucun doublon exact entre les trois ensembles.")
print("Aucun fichier original supprimé ou déplacé.")

classe,NORMAL,PNEUMONIA,Total images,Nombre de groupes
ensemble_final,,,,
train,1079,2721,3800,2152
val,269,684,953,539
test,231,387,618,426


Images écartées par précaution : 453
Images d'apprentissage + validation : 4753
Aucun groupe commun entre les trois ensembles.
Aucun doublon exact entre les trois ensembles.
Aucun fichier original supprimé ou déplacé.


In [8]:
from pathlib import Path
from datetime import datetime
import json
import subprocess
import sys
import pandas as pd

if "repartition" not in globals():
    raise RuntimeError("Exécute d'abord la cellule de répartition.")

DOSSIER_PROJET = Path(
    r"C:\Users\David KUNYIMA\OneDrive\Desktop\Memoire_CNN_Attention"
)
DOSSIER_IMAGES = DOSSIER_PROJET / "archive" / "chest_xray"

# Un dossier daté évite d'écraser une sauvegarde précédente.
horodatage = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
DOSSIER_SAUVEGARDE = (
    DOSSIER_PROJET / "repartitions" / f"repartition_{horodatage}"
)
DOSSIER_SAUVEGARDE.mkdir(parents=True, exist_ok=False)

def sauvegarder_tableau(tableau, nom_fichier):
    copie = tableau.copy()

    # Chemins relatifs : utiles si le projet change d'ordinateur.
    if "chemin" in copie.columns:
        copie["chemin_relatif"] = copie["chemin"].apply(
            lambda chemin: Path(chemin)
            .relative_to(DOSSIER_IMAGES)
            .as_posix()
        )
        copie = copie.drop(columns="chemin")

    copie.to_csv(
        DOSSIER_SAUVEGARDE / nom_fichier,
        index=False,
        encoding="utf-8-sig"
    )

# Répartition complète et listes utilisées par les modèles.
sauvegarder_tableau(repartition, "repartition_complete.csv")

for ensemble in ["train", "val", "test"]:
    selection = repartition[
        repartition["ensemble_final"] == ensemble
    ]
    sauvegarder_tableau(selection, f"{ensemble}.csv")

# Traçabilité : conserver l'audit et les raisons des exclusions.
sauvegarder_tableau(audit, "inventaire_complet_avec_groupes.csv")
sauvegarder_tableau(exclusions_df, "exclusions_precaution.csv")
sauvegarder_tableau(copies_ecartees, "copies_exactes_ecartees.csv")

parametres = {
    "graine_aleatoire": 42,
    "fraction_validation_par_groupes": 0.20,
    "regroupement": (
        "Identifiants présumés extraits des noms, reliés par les "
        "doublons exacts de pixels après conversion RGB."
    ),
    "limite": "Identité clinique des patients non vérifiée.",
    "politique_test": (
        "Test original dédupliqué ; exclusion hors test des groupes "
        "également présents dans le test."
    ),
    "effectifs": {
        ensemble: int((repartition["ensemble_final"] == ensemble).sum())
        for ensemble in ["train", "val", "test"]
    },
    "images_exclues_par_precaution": int(len(exclusions_df)),
    "version_python": sys.version
}

with open(
    DOSSIER_SAUVEGARDE / "parametres.json",
    "w",
    encoding="utf-8"
) as fichier:
    json.dump(parametres, fichier, indent=2, ensure_ascii=False)

# Conserver les versions des bibliothèques installées.
versions = subprocess.check_output(
    [sys.executable, "-m", "pip", "freeze"],
    text=True
)
(DOSSIER_SAUVEGARDE / "requirements.txt").write_text(
    versions, encoding="utf-8"
)

# Vérifier que la répartition sauvegardée se recharge correctement.
verification = pd.read_csv(
    DOSSIER_SAUVEGARDE / "repartition_complete.csv"
)

assert len(verification) == len(repartition)
assert verification["chemin_relatif"].is_unique
assert set(verification["empreinte"]) == set(repartition["empreinte"])

print("Sauvegarde terminée et vérifiée.")
print("Images enregistrées dans les listes :", len(verification))
print("Dossier :", DOSSIER_SAUVEGARDE)

Sauvegarde terminée et vérifiée.
Images enregistrées dans les listes : 5371
Dossier : C:\Users\David KUNYIMA\OneDrive\Desktop\Memoire_CNN_Attention\repartitions\repartition_20260917_102959_713272
